# Testing at Multiple Layers

Testing at multiple layers assigns a failure to the smallest useful scope. Unit tests can validate a pure record validator. Integration tests can prove that accepted records and their request identifier commit together in a real database. A browser or CLI journey can check the public interface. Passing one layer does not establish the others.

Consider an inventory import requiring id, owner, and count. Parsing answers whether input is valid JSON. Schema validation answers whether it has the required structure and types. Domain validation checks such rules as a nonnegative integer count and unique identifiers. These checks do not prove that the inventory matches physical reality.

Validate a complete batch before making it visible. In the downloadable SQLite example, insertion and the request receipt share one transaction. A duplicate request with the same content returns the prior result; reuse of the request identifier with different content is rejected. A database conflict rolls back the batch. This protects against partial writes and careless retries without claiming exactly-once delivery across arbitrary distributed systems.

Use malformed JSON, a string instead of an integer, missing fields, unexpected fields, duplicate IDs, and a database conflict as distinct test cases. Use generated cases to check invariants, but do not replace concrete examples and independent expected results with tests that simply repeat the implementation. The browser companion performs parsing and validation only; persistence is exercised by the downloadable application.

## Predict before running

A1/Operations/3 and A2/Research/0 form a valid two-record batch. Changing A2.count to −1 rejects the entire batch. Changing it to the string "3" also rejects it: permissive coercion could conceal upstream schema errors. A successful validation is not a database commit.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import foundations as f

In [ ]:
rows=[{'id':'A1','owner':'Operations','count':3},{'id':'A2','owner':'Research','count':0}]
db=f.database()
try:
    try:
        f.ingest(db,'batch-1',rows,fail_after=1)
    except RuntimeError:
        pass
    assert db.execute('SELECT COUNT(*) FROM inventory').fetchone()[0]==0
    print('First commit:', f.ingest(db,'batch-1',rows))
    print('Retry:', f.ingest(db,'batch-1',rows))
    assert db.execute('SELECT COUNT(*) FROM inventory').fetchone()[0]==2
    try:
        f.ingest(db,'batch-1',rows[:1])
    except ValueError as error:
        print('Expected conflict:',error)
    else:
        raise AssertionError('A conflicting request must fail')
finally:
    db.close()

## Investigate

1. Calculate the default two-record batch. Expect two accepted records and zero validation errors.
2. Set one count to −1. Expect zero accepted records.
3. Restore the count, then duplicate an id. Expect atomic rejection.
4. Try invalid JSON and an unexpected field; distinguish syntax from schema errors.
5. Run the downloadable SQLite tests, including a second-row conflict and request-id replay.
6. Export a valid browser result and explain why it is validation evidence rather than persistence evidence.

## Integration and extension

Write a test matrix for a CSV-to-JSON inventory import. Include syntax, identity, escaping, duplicate requests, database conflicts, and a corrected-input retry.

Explore concurrent writers, transaction isolation, and the difference between at-least-once delivery and idempotent application behavior.

Record assumptions, your changed input, its result, and what you can conclude.

## Primary evidence

- [Python json documentation — decoding and exceptions](https://docs.python.org/3/library/json.html)
- [SQLite — Atomic Commit](https://www.sqlite.org/atomiccommit.html)
- [SQLite — Transaction](https://www.sqlite.org/lang_transaction.html)